# Чекпойнт 4: Baseline-модели и метрики

**ВКР:** Детекция дублей в аналитических таксономиях методами NLP

## Содержание
1. [Метрики: обоснование выбора](#sec1)
2. [Загрузка данных](#sec2)
3. [Feature Engineering — 20 признаков](#sec3)
4. [B0: Random baseline](#sec4)
5. [B1/B2: Threshold на строковом сходстве](#sec5)
6. [B3: TF-IDF + Logistic Regression](#sec6)
7. [B4: TF-IDF + Linear SVM](#sec7)
8. [B5: Feature Engineering + GBM](#sec8)
9. [Сравнение по типам дублей](#sec9)
10. [Hard Semantic Test Set](#sec10)
11. [Выводы](#sec11)

---
## 1. Метрики: обоснование выбора <a id='sec1'></a>

### Почему не Accuracy

Датасет сбалансирован 50/50 в обучающей выборке, однако в **продакшен-условиях дубли редки** — реальный дисбаланс 1:5 и выше. Accuracy при таком дисбалансе завышена тривиальным классификатором «всё не дубль». Поэтому Accuracy не используется.

### Выбранный набор метрик

| Метрика | Роль |
|---------|------|
| **F1-score** | Основная. Баланс Precision и Recall. Стандарт задачи (QQP, DITTO, OAEI) |
| **Precision** | Доля верных предупреждений — влияет на доверие аналитика |
| **Recall** | Полнота — пропущенный дубль накапливается незаметно |
| **AUC-ROC** | Ранжирование независимо от порога — для сравнения моделей |
| **F1 по dup_type** | Диагностика: какие типы дублей модель умеет находить |

**Порог θ** подбирается на val-выборке через grid search по `[0.10, 0.99]`, а не фиксируется на 0.5.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re, difflib, warnings, json, time
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              roc_auc_score, confusion_matrix,
                              average_precision_score, precision_recall_curve)
warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3,
})

TYPE_ORDER = ['exact','typo','case','synonym','permutation','semantic','not_duplicate']
PALETTE    = {'exact':'#1abc9c','typo':'#3498db','case':'#9b59b6',
              'synonym':'#e67e22','permutation':'#e74c3c',
              'semantic':'#c0392b','not_duplicate':'#95a5a6'}
print('OK')

---
## 2. Загрузка данных <a id='sec2'></a>

In [ ]:
DATA = 'data/'
HARD = 'data/pairs_hard_semantic_test.csv'

df_train = pd.read_csv(DATA + 'pairs_train.csv')
df_val   = pd.read_csv(DATA + 'pairs_val.csv')
df_test  = pd.read_csv(DATA + 'pairs_test.csv')
df_hard  = pd.read_csv(HARD)

print(f'Train: {len(df_train):,}  Val: {len(df_val):,}  Test: {len(df_test):,}  Hard: {len(df_hard)}')
print(f'Duplicate rate — Train: {df_train.label.mean():.3f}  Test: {df_test.label.mean():.3f}')
print(f'Hard set — label=1: {(df_hard.label==1).sum()}  label=0: {(df_hard.label==0).sum()}')

---
## 3. Feature Engineering — 20 признаков <a id='sec3'></a>

Признаки разделены на три группы:

| Группа | Признаки | Что улавливают |
|--------|----------|----------------|
| **Similarity** | char_sim, norm_sim, jaccard, overlap, sim×jaccard, sim×overlap, sim−jaccard | Строковое и токенное сходство |
| **Length/Token** | len_1/2, tok_1/2, len_diff, tok_diff, len_ratio, tok_ratio | Структурные размеры |
| **Structure** | same_action, same_domain, prefix_tok2/3, common_prefix_len | Семантическая структура |

**Ключевой признак:** `sim_x_overlap = norm_sim × overlap` — произведение сходств строк и токенного перекрытия. Оказывается наиболее значимым для GBM (~95% важности) — сигнал о вырожденности задачи на синтетическом датасете.

In [ ]:
def token_overlap(a, b):
    ta, tb = set(str(a).split('_')), set(str(b).split('_'))
    return len(ta & tb) / max(len(ta), len(tb), 1)

def shared_prefix_tokens(a, b, n):
    ta, tb = str(a).split('_'), str(b).split('_')
    k = 0
    for i in range(min(n, len(ta), len(tb))):
        if ta[i] == tb[i]: k += 1
        else: break
    return k

def common_prefix_len(a, b):
    a, b = str(a), str(b)
    i = 0
    while i < min(len(a), len(b)) and a[i] == b[i]: i += 1
    return i

def build_features(df):
    e1 = df['event_1'].fillna('').astype(str)
    e2 = df['event_2'].fillna('').astype(str)
    has = lambda c: c in df.columns
    f = pd.DataFrame()
    # Similarity
    f['char_sim']          = df['char_sim']  if has('char_sim')  else 0.0
    f['norm_sim']          = df['norm_sim']  if has('norm_sim')  else 0.0
    f['jaccard']           = df['jaccard']   if has('jaccard')   else 0.0
    f['overlap']           = [token_overlap(a,b) for a,b in zip(e1,e2)]
    f['sim_x_jaccard']     = f['norm_sim'] * f['jaccard']
    f['sim_x_overlap']     = f['norm_sim'] * f['overlap']
    f['sim_minus_jaccard'] = f['norm_sim'] - f['jaccard']
    # Length/Token
    f['len_1']    = df['len_1']   if has('len_1')   else e1.apply(len)
    f['len_2']    = df['len_2']   if has('len_2')   else e2.apply(len)
    f['len_diff'] = df['len_diff'] if has('len_diff') else (f['len_1']-f['len_2']).abs()
    f['tok_1']    = df['tok_1']   if has('tok_1')   else e1.apply(lambda x: len(x.split('_')))
    f['tok_2']    = df['tok_2']   if has('tok_2')   else e2.apply(lambda x: len(x.split('_')))
    f['tok_diff'] = df['tok_diff'] if has('tok_diff') else (f['tok_1']-f['tok_2']).abs()
    f['len_ratio']= f[['len_1','len_2']].min(axis=1) / (f[['len_1','len_2']].max(axis=1)+1)
    f['tok_ratio']= f[['tok_1','tok_2']].min(axis=1) / (f[['tok_1','tok_2']].max(axis=1)+1)
    # Structure
    f['same_action']       = df['same_action'] if has('same_action') else 0
    f['same_domain']       = df['same_domain'] if has('same_domain') else 0
    f['prefix_tok2']       = [shared_prefix_tokens(a,b,2) for a,b in zip(e1,e2)]
    f['prefix_tok3']       = [shared_prefix_tokens(a,b,3) for a,b in zip(e1,e2)]
    f['common_prefix_len'] = [common_prefix_len(a,b)      for a,b in zip(e1,e2)]
    return f.fillna(0).astype(float)

t0 = time.time()
X_train = build_features(df_train); y_train = df_train['label'].values
X_val   = build_features(df_val);   y_val   = df_val['label'].values
X_test  = build_features(df_test);  y_test  = df_test['label'].values
X_hard  = build_features(df_hard);  y_hard  = df_hard['label'].values
print(f'{X_train.shape[1]} признаков построено за {time.time()-t0:.1f}s')
print('Признаки:', list(X_train.columns))

In [ ]:
# Корреляция признаков с label
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Feature Engineering: анализ признаков', fontsize=12, fontweight='bold')

ax = axes[0]
corr_l = X_train.corrwith(pd.Series(y_train, name='label')).sort_values()
c = ['#e74c3c' if v<0 else '#2ecc71' for v in corr_l.values]
corr_l.plot(kind='barh', ax=ax, color=c, edgecolor='white')
ax.set_title('Корреляция признаков с label (Pearson r)')
ax.set_xlabel('r'); ax.axvline(0, color='black', lw=0.8)

ax = axes[1]
corr_m = X_train.assign(label=y_train).corr()
mask = np.triu(np.ones_like(corr_m, dtype=bool))
sns.heatmap(corr_m, mask=mask, annot=False, cmap='RdBu_r', center=0,
            ax=ax, linewidths=0.3, vmin=-1, vmax=1)
ax.set_title('Корреляционная матрица признаков')

plt.tight_layout()
plt.savefig('figures/feat_correlation.png', dpi=120, bbox_inches='tight')
plt.show()
print('Рис. 1 сохранён.')

In [ ]:
# Вспомогательные функции
results_test = {}
results_hard = {}

def evaluate(name, y_true, y_pred, y_score, df_ref=None):
    P   = precision_score(y_true, y_pred, zero_division=0)
    R   = recall_score(y_true, y_pred, zero_division=0)
    F1  = f1_score(y_true, y_pred, zero_division=0)
    try:   AUC = roc_auc_score(y_true, y_score)
    except: AUC = 0.5
    print(f'  {name:40s}  P={P:.3f}  R={R:.3f}  F1={F1:.3f}  AUC={AUC:.3f}')
    return {'P':round(P,4),'R':round(R,4),'F1':round(F1,4),'AUC':round(AUC,4)}

def find_best_theta(y_true, scores, thetas):
    best_f1, best_t = 0, 0.5
    for t in thetas:
        f = f1_score(y_true, (scores >= t).astype(int), zero_division=0)
        if f > best_f1: best_f1, best_t = f, t
    return best_t, best_f1

thetas = np.arange(0.10, 0.99, 0.01)
print('Вспомогательные функции готовы.')

---
## 4. B0: Random baseline <a id='sec4'></a>

In [ ]:
print('=== B0: Random ===')
np.random.seed(42)
y_b0 = (np.random.rand(len(y_test)) < y_train.mean()).astype(int)
results_test['B0: Random'] = evaluate('B0: Random', y_test, y_b0, np.random.rand(len(y_test)))

---
## 5. B1/B2: Threshold на строковом сходстве <a id='sec5'></a>

In [ ]:
# Калибровка порога на val-выборке
f1_char_curve = [f1_score(y_val,(df_val['char_sim']>=t).astype(int),zero_division=0) for t in thetas]
f1_norm_curve = [f1_score(y_val,(df_val['norm_sim']>=t).astype(int),zero_division=0) for t in thetas]
bt_char, _ = thetas[np.argmax(f1_char_curve)], max(f1_char_curve)
bt_norm, _ = thetas[np.argmax(f1_norm_curve)], max(f1_norm_curve)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thetas, f1_char_curve, color='#3498db', lw=2,
        label=f'char_sim (best θ={bt_char:.2f}, F1={max(f1_char_curve):.3f})')
ax.plot(thetas, f1_norm_curve, color='#2ecc71', lw=2,
        label=f'norm_sim (best θ={bt_norm:.2f}, F1={max(f1_norm_curve):.3f})')
ax.axvline(bt_char, color='#3498db', ls='--', alpha=0.5)
ax.axvline(bt_norm, color='#2ecc71', ls='--', alpha=0.5)
ax.set_xlabel('Порог θ'); ax.set_ylabel('F1 (val-выборка)')
ax.set_title('Калибровка порога на val-выборке'); ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig('figures/threshold_calibration.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'=== B1: char_sim (θ={bt_char:.2f}) ===')
y_b1 = (df_test['char_sim'].values >= bt_char).astype(int)
results_test[f'B1: char_sim (θ={bt_char:.2f})'] = evaluate(
    f'B1: char_sim (θ={bt_char:.2f})', y_test, y_b1, df_test['char_sim'].values)

print(f'=== B2: norm_sim (θ={bt_norm:.2f}) ===')
y_b2 = (df_test['norm_sim'].values >= bt_norm).astype(int)
results_test[f'B2: norm_sim (θ={bt_norm:.2f})'] = evaluate(
    f'B2: norm_sim (θ={bt_norm:.2f})', y_test, y_b2, df_test['norm_sim'].values)

---
## 6. B3: TF-IDF + Logistic Regression <a id='sec6'></a>

In [ ]:
def concat_pair(df):
    return (df['event_1'].fillna('') + ' [SEP] ' + df['event_2'].fillna('')).tolist()

X_tr_txt = concat_pair(df_train); X_te_txt = concat_pair(df_test)
X_vl_txt = concat_pair(df_val);   X_hd_txt = concat_pair(df_hard)

TFIDF = dict(analyzer='char_wb', ngram_range=(2,4), max_features=50000, sublinear_tf=True)

print('=== B3: TF-IDF + Logistic Regression ===')
t0 = time.time()
pipe_lr = Pipeline([
    ('tfidf', TfidfVectorizer(**TFIDF)),
    ('clf',   LogisticRegression(C=1.0, max_iter=1000, random_state=42, class_weight='balanced'))
])
pipe_lr.fit(X_tr_txt, y_train)
print(f'  Обучено за {time.time()-t0:.1f}s')
y_b3 = pipe_lr.predict(X_te_txt)
results_test['B3: TF-IDF + LogReg'] = evaluate(
    'B3: TF-IDF + LogReg', y_test, y_b3, pipe_lr.predict_proba(X_te_txt)[:,1])

---
## 7. B4: TF-IDF + Linear SVM <a id='sec7'></a>

In [ ]:
print('=== B4: TF-IDF + LinearSVM ===')
t0 = time.time()
pipe_svm = Pipeline([
    ('tfidf', TfidfVectorizer(**TFIDF)),
    ('clf',   CalibratedClassifierCV(
                  LinearSVC(C=1.0, max_iter=3000, random_state=42, class_weight='balanced'), cv=3))
])
pipe_svm.fit(X_tr_txt, y_train)
print(f'  Обучено за {time.time()-t0:.1f}s')
y_b4 = pipe_svm.predict(X_te_txt)
results_test['B4: TF-IDF + SVM'] = evaluate(
    'B4: TF-IDF + SVM', y_test, y_b4, pipe_svm.predict_proba(X_te_txt)[:,1])

---
## 8. B5: Feature Engineering + Gradient Boosting <a id='sec8'></a>

In [ ]:
print('=== B5: Feature Engineering + GBM ===')
scaler = StandardScaler()
Xtr_sc = scaler.fit_transform(X_train); Xte_sc = scaler.transform(X_test)
Xvl_sc = scaler.transform(X_val);       Xhd_sc = scaler.transform(X_hard)

t0 = time.time()
gb = GradientBoostingClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8, random_state=42)
gb.fit(Xtr_sc, y_train)
print(f'  Обучено за {time.time()-t0:.1f}s')
y_b5 = gb.predict(Xte_sc)
results_test['B5: GBM (features)'] = evaluate(
    'B5: GBM (features)', y_test, y_b5, gb.predict_proba(Xte_sc)[:,1])

# Feature importance
fi = pd.Series(gb.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print('\nТоп-10 признаков:')
print(fi.head(10).round(4).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
fi_top = fi.head(15).sort_values()
colors_fi = ['#e74c3c' if any(k in n for k in ['sim','jaccard','overlap'])
             else '#3498db' if any(k in n for k in ['len','tok'])
             else '#2ecc71' for n in fi_top.index]
fi_top.plot(kind='barh', ax=ax, color=colors_fi, edgecolor='white')
ax.set_title('GBM: важность признаков (топ-15)', fontsize=12)
ax.set_xlabel('Feature importance')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#e74c3c',label='similarity'),
                   Patch(color='#3498db',label='length/token'),
                   Patch(color='#2ecc71',label='structure')], fontsize=9)
plt.tight_layout(); plt.savefig('figures/feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 9. Сравнение по типам дублей <a id='sec9'></a>

In [ ]:
models_preds = {
    'B1: char_sim':   y_b1, 'B2: norm_sim':  y_b2,
    'B3: TF-IDF+LR': y_b3, 'B4: TF-IDF+SVM':y_b4, 'B5: GBM': y_b5,
}
rows = []
for t in TYPE_ORDER:
    mask = (df_test['dup_type'].values == t)
    if mask.sum() < 2: continue
    row = {'Тип': t, 'n': int(mask.sum())}
    for mn, yp in models_preds.items():
        row[mn] = round(f1_score(y_test[mask], yp[mask], zero_division=0), 3)
    rows.append(row)
df_bt = pd.DataFrame(rows)
print(df_bt.to_string(index=False))

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(df_bt.set_index('Тип')[list(models_preds.keys())].T,
            annot=True, fmt='.3f', cmap='RdYlGn',
            ax=ax, linewidths=0.5, vmin=0, vmax=1, annot_kws={'size':11})
ax.set_title('F1 по типам дублей — все модели', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.savefig('figures/f1_by_type.png', dpi=120, bbox_inches='tight')
plt.show(); print('Рис. сохранён.')

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Confusion Matrix — baseline-модели (тест)', fontsize=13, fontweight='bold')
cms = [(f'B0: Random', y_b0),
       (f'B1: char_sim θ={bt_char:.2f}', y_b1),
       (f'B2: norm_sim θ={bt_norm:.2f}', y_b2),
       ('B3: TF-IDF+LR', y_b3), ('B4: TF-IDF+SVM', y_b4), ('B5: GBM', y_b5)]
for ax, (name, yp) in zip(axes.flat, cms):
    cm = confusion_matrix(y_test, yp)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Не дубль','Дубль'],
                yticklabels=['Не дубль','Дубль'],
                linewidths=0.5, cbar=False, annot_kws={'size':11})
    ax.set_title(f'{name}  F1={f1_score(y_test,yp,zero_division=0):.3f}', fontsize=10)
    ax.set_xlabel('Предсказано'); ax.set_ylabel('Истинно')
plt.tight_layout(); plt.savefig('figures/confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 10. Hard Semantic Test Set <a id='sec10'></a>

Специальный тест для проверки способности моделей к **настоящей семантической детекции**.

- `label=1` (74 пары): `login_success` / `user_authenticated`, char_sim ≈ 0.19
- `label=0` (40 пар): антонимы-ловушки `card_blocked` / `card_unblocked`, char_sim ≈ 0.92

Модели, не понимающие семантику, покажут F1 ≈ 0.

In [ ]:
hard_preds = {
    'B1: char_sim':   (df_hard['char_sim'].values >= bt_char).astype(int),
    'B2: norm_sim':   (df_hard['norm_sim'].values >= bt_norm).astype(int),
    'B3: TF-IDF+LR':  pipe_lr.predict(X_hd_txt),
    'B4: TF-IDF+SVM': pipe_svm.predict(X_hd_txt),
    'B5: GBM':        gb.predict(Xhd_sc),
}
hard_scores = {
    'B1: char_sim':   df_hard['char_sim'].values,
    'B2: norm_sim':   df_hard['norm_sim'].values,
    'B3: TF-IDF+LR':  pipe_lr.predict_proba(X_hd_txt)[:,1],
    'B4: TF-IDF+SVM': pipe_svm.predict_proba(X_hd_txt)[:,1],
    'B5: GBM':        gb.predict_proba(Xhd_sc)[:,1],
}
hard_f1, hard_auc = {}, {}
print('=== Hard Semantic Test Set ===')
for mn in hard_preds:
    f1  = f1_score(y_hard, hard_preds[mn], zero_division=0)
    try: auc = roc_auc_score(y_hard, hard_scores[mn])
    except: auc = 0.5
    hard_f1[mn]=round(f1,3); hard_auc[mn]=round(auc,3)
    print(f'  {mn:35s}  F1={f1:.3f}  AUC={auc:.3f}')

In [ ]:
MODEL_KEYS = ['B1: char_sim','B2: norm_sim','B3: TF-IDF+LR','B4: TF-IDF+SVM','B5: GBM']
TEST_F1 = {
    'B1: char_sim':  results_test[f'B1: char_sim (θ={bt_char:.2f})']['F1'],
    'B2: norm_sim':  results_test[f'B2: norm_sim (θ={bt_norm:.2f})']['F1'],
    'B3: TF-IDF+LR': results_test['B3: TF-IDF + LogReg']['F1'],
    'B4: TF-IDF+SVM':results_test['B4: TF-IDF + SVM']['F1'],
    'B5: GBM':       results_test['B5: GBM (features)']['F1'],
}

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(MODEL_KEYS)); w = 0.35
b1 = ax.bar(x-w/2, [TEST_F1[k] for k in MODEL_KEYS], w,
            label='Основной тест', color='#2ecc71', edgecolor='white')
b2 = ax.bar(x+w/2, [hard_f1[k]   for k in MODEL_KEYS], w,
            label='Hard Semantic Set', color='#e74c3c', edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels([k.replace(' ',r'\n') for k in MODEL_KEYS], fontsize=9)
ax.set_title('F1: Основной тест vs Hard Semantic Set\n(обвал = нет семантического понимания)',
             fontsize=12, fontweight='bold')
ax.set_ylabel('F1-score'); ax.set_ylim(0,1.18); ax.legend(fontsize=10)
for b,v in zip(b1,[TEST_F1[k] for k in MODEL_KEYS]):
    ax.text(b.get_x()+b.get_width()/2, v+0.01, f'{v:.3f}', ha='center', fontsize=8)
for b,v in zip(b2,[hard_f1[k] for k in MODEL_KEYS]):
    c = '#c0392b' if v<0.5 else '#e67e22'
    ax.text(b.get_x()+b.get_width()/2, v+0.01, f'{v:.3f}',
            ha='center', fontsize=9, color=c, fontweight='bold')
plt.tight_layout(); plt.savefig('figures/hard_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 11. Выводы <a id='sec11'></a>

### 11.1 Результаты на основном тесте

| Модель | F1 | Precision | Recall | AUC |
|--------|----|-----------|--------|-----|
| B0: Random | 0.495 | 0.492 | 0.497 | 0.501 |
| B1: char_sim (θ=0.58) | 0.962 | 0.991 | 0.935 | 0.958 |
| B2: norm_sim (θ=0.58) | 0.984 | 0.991 | 0.976 | 0.998 |
| B3: TF-IDF + LogReg | 0.988 | 0.988 | 0.988 | 0.999 |
| **B4: TF-IDF + SVM** | **0.998** | 0.997 | 0.998 | 1.000 |
| B5: GBM (features) | 0.998 | 0.997 | 0.998 | 1.000 |

### 11.2 Hard Semantic Test Set — главный результат

| Модель | F1 (тест) | F1 (hard) | Обвал |
|--------|-----------|-----------|-------|
| B1: char_sim | 0.962 | **0.059** | −90% |
| B2: norm_sim | 0.984 | **0.059** | −93% |
| B3: TF-IDF+LR | 0.988 | **0.787** | −20% |
| **B4: TF-IDF+SVM** | **0.998** | **0.774** | −22% |
| B5: GBM | 0.998 | **0.222** | −78% |

### 11.3 Интерпретация

**Три класса поведения:**

1. **Строковые пороги (B1/B2):** F1 = 0.06 на hard set — полный провал. `login_success` / `user_authenticated` (char_sim=0.19) не обнаруживаются, зато `card_blocked` / `card_unblocked` (char_sim=0.92) ложно принимаются за дубли.

2. **TF-IDF + ML (B3/B4):** F1 ≈ 0.77–0.79 на hard set. Символьные 4-граммы (`_success`, `_view`, `_click`) улавливают морфологические паттерны действий. **Лучший baseline** по качество/интерпретируемость: B4 TF-IDF+SVM.

3. **GBM на ручных признаках (B5):** F1 = 0.22 на hard set. Модель переобучилась под `sim_x_overlap` (95% важности) — при семантических парах этот признак близок к нулю, и предсказание деградирует.

### 11.4 Вывод для следующего шага

Потолок всех baseline-методов на семантически сложных парах — **F1 ≈ 0.79** (TF-IDF+SVM). Для преодоления необходимы **предобученные языковые модели (SBERT)**, которые кодируют смысл слов в векторном пространстве.

| Следующий шаг | Ожидаемый F1 (hard set) |
|---|---|
| TF-IDF + SVM (текущий потолок) | ~0.77 |
| SBERT `all-MiniLM-L6-v2` | ~0.85–0.90 |
| SBERT `all-mpnet-base-v2` | ~0.88–0.93 |
| Cross-encoder (DITTO) | ~0.92–0.96 |